# Aluminum Fermi surface (JVASP-816) with SlakoNet

A standalone walkthrough: load aluminum from JARVIS-DFT, sample band energies on a
full Brillouin-zone grid with a SlakoNet tight-binding model, and extract + plot the
Fermi surface. No web service or credentials, just the `fermisurf` scripts.

Run from the repo root so `import fermisurf` resolves, in an environment with
`slakonet`, `torch`, `scikit-image`, `plotly`, and `jarvis-tools`.

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so `import fermisurf` works from examples/

from slakonet.optim import get_atoms

atoms = get_atoms("JVASP-816")[0]
print(atoms.composition.reduced_formula, "| atoms:", atoms.num_atoms,
      "| a =", round(atoms.lattice.abc[0], 3), "A")

## Band energies on a Brillouin-zone grid

`eigenfield` runs the SlakoNet model on an `nk x nk x nk` Monkhorst-Pack grid and
returns the band energies as an `(nx, ny, nz, nbands)` array in eV, the Fermi energy,
and the real-space cell. Denser `nk` gives a smoother surface at more cost; CPU is the
default. `nk=20` takes well under a minute for a 1-atom cell.

In [ ]:
from fermisurf.slakonet_fermi import eigenfield

field, efermi, cell = eigenfield(atoms, nk=20, device="cpu")

nb = field.shape[3]
crossing = [b for b in range(nb) if field[..., b].min() < efermi < field[..., b].max()]
print("field", field.shape, "| E_F =", round(efermi, 3), "eV")
print("bands crossing E_F:", crossing)

## Extract and plot the Fermi surface

`fermi_surface_figure` marching-cubes the `E = E_F` isosurface of every band that
crosses the Fermi level, maps it from fractional to Cartesian reciprocal space, and
returns an interactive Plotly figure. `fig.show()` renders it inline; `save` writes a
self-contained HTML.

In [ ]:
from fermisurf.isosurface import fermi_surface_figure, save

fig = fermi_surface_figure(
    field, efermi, cell,
    title="Aluminum (JVASP-816) Fermi surface - SlakoNet")
fig.show()

In [ ]:
save(fig, "JVASP-816_Al_fermi.html")
print("wrote JVASP-816_Al_fermi.html - open it in any browser to rotate the surface")

## From a VASP calculation instead

With a static VASP run on a dense, non-symmetry-reduced k-mesh (`ISYM=0`), the VASP
front-end does the same via IFermi:

```python
from fermisurf.vasp_fermi import build_surface
from ifermi.plot import FermiSurfacePlotter, save_plot

fs = build_surface("vasprun.xml", interpolation_factor=5)
save_plot(FermiSurfacePlotter(fs).get_plot(plot_type="plotly"), "fs.html")
```

or from the command line:

```bash
python -m fermisurf.vasp_fermi --vasprun vasprun.xml --out fs.html
```